In [ ]:
%pip install  openpyxl

In [1]:
import pandas as pd
import openpyxl
from scipy.stats import chi2_contingency
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import  f_oneway



In [ ]:
df1=pd.read_excel(r"C:\Users\Lenovo\Downloads\credit_card\case_study1.xlsx")
df2=pd.read_excel(r"C:\Users\Lenovo\Downloads\credit_card\case_study2.xlsx")


In [ ]:
def clean_mostly_null_cols(df):
    a=[]
    for i in df.columns:
        if df2[df2[i]==-99999].shape[0]>10000:
            a.append(i)
    return df.drop(a,axis=1)

In [ ]:
def clean_mostly_null_rows(df):
    for i in df.columns:
        df=df[df['i']!=-99999]
    return df

    

In [ ]:
def find_merge_cols(df1,df2):
    merge_cols=[]
    for i in df1.columns.tolist():
        if i in df2.columns.tolist():
            merge_cols.append(i)
    return merge_cols


In [ ]:
def chi_suqure_test_removal_cols(df,target):
    ##only valid when your .. target is object or cat columns
    object_arr=[]
    cols_to_remove=[]
    df1=df.drop(target,axis=1)
    for i in df1.columns:
        if df1[i].dtype=='object':
                object_arr.append(i)
    for i in object_arr:
         chi2,pval,_,_=chi2_contingency(pd.crosstab(df[i],df[target]))
         if pval>0.05:
              cols_to_remove.append(i)
    return df.drop(cols_to_remove,axis=1),object_arr
         
        



In [ ]:
def column_filter_based_on_vif_and_annova(df,target):
    numeric_cols=[]
    for i in df.columns:
        if df[i].dtype!='object' and i not in ['Approved_Flag','PROSPECTID']:
            numeric_cols.append(i)
    vif_data=df[numeric_cols]
    total_columns=vif_data.shape[1]
    columns_to_kept=[]
    columns_index=0
    for i in range(0,total_columns):
        vif_value=variance_inflation_factor(vif_data,columns_index)
        print(vif_value,'___',columns_index)
        if vif_value<=6:
            columns_to_kept.append(numeric_cols[i])
            columns_index=columns_index+1
        else:
            vif_data=vif_data.drop([numeric_cols[i]],axis=1)
    annova_result={}
    columns_to_kept_in_numerical=[]
    for i in columns_to_kept:
        groups=[df[df['Approved_Flag']==cat][i].dropna() for cat in df['Approved_Flag'].unique()]
        f_stat,p_value=f_oneway(*groups)
        annova_result[i]=p_value
    anova_df = pd.DataFrame(list(annova_result.items()), columns=['feature', 'p_value'])
    anova_df = anova_df.sort_values('p_value')
    return anova_df[anova_df['p_value']<=0.05]['feature'].unique().tolist()

In [ ]:

df=df1.merge(df2,on=find_merge_cols(df1,df2),how='inner')

In [2]:
target='Approved_Flag'

In [ ]:
df,obj_arr=chi_suqure_test_removal_cols(df,target=target)
final_numeric_cols=column_filter_based_on_vif_and_annova(df,target=target)
feature=final_numeric_cols+obj_arr
feature = [f for f in feature if f != 'Credit_Score']

In [ ]:
df=df[feature+target]


In [ ]:
def education_encoder(df,col):
    education_level_map = {
    'SSC': 1,
    '12TH': 2,
    'UNDER GRADUATE': 3,
    'GRADUATE': 3,
    'PROFESSIONAL': 3,
    'POST-GRADUATE': 4,
    'OTHERS': 1
}
   
    df[col] = df[col].map(education_level_map)
    df[col]=df[col].astype('int64')
    return df



In [ ]:
##encoding of education column
def encoder_cols(df,col:list[str]) -> pd.DataFrame:
    return pd.get_dummies(df,columns=col)

